In [1]:
import pandas as pd
import numpy as np
import math

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
DATA_PATH = "data/processed/clean_products.csv"

df = pd.read_csv(DATA_PATH)
pd.set_option("display.max_columns", None)  
print("Shape:", df.shape)
df.head()

Shape: (1351, 23)


,product_id,product_name,category_path,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,product_link,cat_level_1,cat_level_2,cat_level_3,cat_level_4,cat_level_5,cat_level_6,cat_level_7,product_name_clean,about_product_clean,category_text,clean_text_no_category,clean_text_with_category
0,B002PD61Y4,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,Computers&Accessories|NetworkingDevices|Networ...,507.0,1208.0,58.0,4.1,8131.0,Connects your computer to a high-speed wireles...,https://m.media-amazon.com/images/I/31+NwZ8gb1...,https://www.amazon.in/D-Link-DWA-131-Wireless-...,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,d link dwa 131 300 mbps wireless nano usb adap...,connects your computer to a high speed wireles...,computers and accessories networkingdevices ne...,d link dwa 131 300 mbps wireless nano usb adap...,d link dwa 131 300 mbps wireless nano usb adap...
1,B002SZEOLG,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,Computers&Accessories|NetworkingDevices|Networ...,749.0,1339.0,44.0,4.2,179692.0,150 Mbps Wi-Fi —— Exceptional wireless speed u...,https://m.media-amazon.com/images/I/31Wb+A3VVd...,https://www.amazon.in/TP-Link-TL-WN722N-150Mbp...,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,tp link nano usb wifi dongle 150mbps high gain...,150 mbps wi fi exceptional wireless speed up t...,computers and accessories networkingdevices ne...,tp link nano usb wifi dongle 150mbps high gain...,tp link nano usb wifi dongle 150mbps high gain...
2,B003B00484,Duracell Plus AAA Rechargeable Batteries (750 ...,Electronics|GeneralPurposeBatteries&BatteryCha...,399.0,499.0,20.0,4.3,27201.0,Duracell Rechargeable AAA 750mAh batteries sta...,https://m.media-amazon.com/images/I/418YrbHVLC...,https://www.amazon.in/Duracell-AAA-750mAh-Rech...,Electronics,GeneralPurposeBatteries&BatteryChargers,RechargeableBatteries,Unknown,Unknown,Unknown,Unknown,duracell plus aaa rechargeable batteries 750 m...,duracell rechargeable aaa 750mah batteries sta...,electronics generalpurposebatteries and batter...,duracell plus aaa rechargeable batteries 750 m...,duracell plus aaa rechargeable batteries 750 m...
3,B003L62T7W,"Logitech B100 Wired USB Mouse, 3 yr Warranty, ...",Computers&Accessories|Accessories&Peripherals|...,279.0,375.0,26.0,4.3,31534.0,"A comfortable, ambidextrous shape feels good i...",https://m.media-amazon.com/images/I/31iFF1Kbkp...,https://www.amazon.in/Logitech-B100-Optical-Mo...,Computers&Accessories,Accessories&Peripherals,"Keyboards,Mice&InputDevices",Mice,Unknown,Unknown,Unknown,logitech b100 wired usb mouse 3 yr warranty 80...,a comfortable ambidextrous shape feels good in...,computers and accessories accessories and peri...,logitech b100 wired usb mouse 3 yr warranty 80...,logitech b100 wired usb mouse 3 yr warranty 80...
4,B004IO5BMQ,"Logitech M235 Wireless Mouse, 1000 DPI Optical...",Computers&Accessories|Accessories&Peripherals|...,699.0,995.0,30.0,4.5,54405.0,You can surf the Web with more comfort and eas...,https://m.media-amazon.com/images/I/31CtVvtFt+...,https://www.amazon.in/Logitech-M235-Wireless-M...,Computers&Accessories,Accessories&Peripherals,"Keyboards,Mice&InputDevices",Mice,Unknown,Unknown,Unknown,logitech m235 wireless mouse 1000 dpi optical ...,you can surf the web with more comfort and eas...,computers and accessories accessories and peri...,logitech m235 wireless mouse 1000 dpi optical ...,logitech m235 wireless mouse 1000 dpi optical ...


In [4]:
df.columns.tolist()

['product_id',
 'product_name',
 'category_path',
 'discounted_price',
 'actual_price',
 'discount_percentage',
 'rating',
 'rating_count',
 'about_product',
 'img_link',
 'product_link',
 'cat_level_1',
 'cat_level_2',
 'cat_level_3',
 'cat_level_4',
 'cat_level_5',
 'cat_level_6',
 'cat_level_7',
 'product_name_clean',
 'about_product_clean',
 'category_text',
 'clean_text_no_category',
 'clean_text_with_category']

In [5]:
def clean_text_value(x):
    if pd.isna(x):
        return ""
    
    x = str(x).strip()
    
    if x.lower() in ["", "nan", "none", "null", "undefined"]:
        return ""
    
    return x


def parse_rating(x):
    try:
        if pd.isna(x):
            return 0.0
        
        x = str(x).strip()
        
        if x.lower() in ["", "nan", "none", "null", "undefined"]:
            return 0.0
        
        return float(x)
    except:
        return 0.0


def parse_rating_count(x):
    try:
        if pd.isna(x):
            return 0
        
        x = str(x).replace(",", "").strip()
        
        if x.lower() in ["", "nan", "none", "null", "undefined"]:
            return 0
        
        return int(float(x))
    except:
        return 0


df["rating_clean"] = df["rating"].apply(parse_rating)
df["rating_count_clean"] = df["rating_count"].apply(parse_rating_count)

df["cat_level_3_clean"] = df["cat_level_3"].apply(clean_text_value)
df["cat_level_4_clean"] = df["cat_level_4"].apply(clean_text_value)


def get_eval_category(row):
    """
    Ưu tiên cat_level_4.
    Nếu không có cat_level_4 thì dùng cat_level_3.
    """
    if row["cat_level_4_clean"] != "":
        return row["cat_level_4_clean"]
    
    if row["cat_level_3_clean"] != "":
        return row["cat_level_3_clean"]
    
    return "Unknown"


df["eval_category"] = df.apply(get_eval_category, axis=1)

df[[
    "product_name",
    "cat_level_3_clean",
    "cat_level_4_clean",
    "eval_category",
    "rating_clean",
    "rating_count_clean"
]].head()

,product_name,cat_level_3_clean,cat_level_4_clean,eval_category,rating_clean,rating_count_clean
0,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,NetworkAdapters,WirelessUSBAdapters,WirelessUSBAdapters,4.1,8131
1,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,NetworkAdapters,WirelessUSBAdapters,WirelessUSBAdapters,4.2,179692
2,Duracell Plus AAA Rechargeable Batteries (750 ...,RechargeableBatteries,Unknown,Unknown,4.3,27201
3,"Logitech B100 Wired USB Mouse, 3 yr Warranty, ...","Keyboards,Mice&InputDevices",Mice,Mice,4.3,31534
4,"Logitech M235 Wireless Mouse, 1000 DPI Optical...","Keyboards,Mice&InputDevices",Mice,Mice,4.5,54405


In [6]:
if "clean_text_no_category" not in df.columns:
    df["clean_text_no_category"] = (
        df["product_name"].fillna("").astype(str) + " " +
        df["about_product"].fillna("").astype(str)
    )

if "clean_text_with_category" not in df.columns:
    df["clean_text_with_category"] = (
        df["product_name"].fillna("").astype(str) + " " +
        df["about_product"].fillna("").astype(str) + " " +
        df["cat_level_3_clean"].fillna("").astype(str) + " " +
        df["cat_level_4_clean"].fillna("").astype(str)
    )

df["clean_text_no_category"] = df["clean_text_no_category"].fillna("").astype(str)
df["clean_text_with_category"] = df["clean_text_with_category"].fillna("").astype(str)

df[["clean_text_no_category", "clean_text_with_category"]].head()

,clean_text_no_category,clean_text_with_category
0,d link dwa 131 300 mbps wireless nano usb adap...,d link dwa 131 300 mbps wireless nano usb adap...
1,tp link nano usb wifi dongle 150mbps high gain...,tp link nano usb wifi dongle 150mbps high gain...
2,duracell plus aaa rechargeable batteries 750 m...,duracell plus aaa rechargeable batteries 750 m...
3,logitech b100 wired usb mouse 3 yr warranty 80...,logitech b100 wired usb mouse 3 yr warranty 80...
4,logitech m235 wireless mouse 1000 dpi optical ...,logitech m235 wireless mouse 1000 dpi optical ...


In [7]:
category_to_indices = {}

for idx, cat in enumerate(df["eval_category"]):
    category_to_indices.setdefault(cat, []).append(idx)


def get_relevant_items(query_idx):
    """
    Ground truth giả lập theo category.
    Relevant items là các sản phẩm khác cùng eval_category.
    """
    cat = df.loc[query_idx, "eval_category"]
    relevant = set(category_to_indices.get(cat, []))
    relevant.discard(query_idx)
    return relevant


# Test thử
sample_idx = 0
print("Product:", df.loc[sample_idx, "product_name"])
print("Category:", df.loc[sample_idx, "eval_category"])
print("Number of relevant items:", len(get_relevant_items(sample_idx)))

Product: D-Link DWA-131 300 Mbps Wireless Nano USB Adapter (Black)
Category: WirelessUSBAdapters
Number of relevant items: 13


In [8]:
vectorizer_no_cat = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

vectorizer_with_cat = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_no_cat = vectorizer_no_cat.fit_transform(df["clean_text_no_category"])
tfidf_with_cat = vectorizer_with_cat.fit_transform(df["clean_text_with_category"])

print("TF-IDF no category:", tfidf_no_cat.shape)
print("TF-IDF with category:", tfidf_with_cat.shape)

TF-IDF no category: (1351, 5000)
TF-IDF with category: (1351, 5000)


In [9]:
def normalize_rating(rating):
    rating = float(rating)
    return max(0.0, min(rating / 5.0, 1.0))


def normalize_popularity(rating_count):
    """
    Dùng log để giảm độ lệch vì rating_count có thể rất lớn.
    """
    value = math.log1p(int(rating_count))
    
    # log1p(100000) khoảng 11.5, nên chia 12 để scale về gần [0,1]
    return max(0.0, min(value / 12.0, 1.0))


rating_scores = df["rating_clean"].apply(normalize_rating).values
popularity_scores = df["rating_count_clean"].apply(normalize_popularity).values

print(rating_scores[:5])
print(popularity_scores[:5])

[0.82 0.84 0.86 0.86 0.9 ]
[0.75029685 1.         0.85092048 0.86323778 0.90868581]


In [10]:
def recommend_content_only(query_idx, top_k=10):
    """
    Model 1:
    Content only = title + description
    Không dùng category.
    """
    sim_scores = cosine_similarity(tfidf_no_cat[query_idx], tfidf_no_cat).flatten()
    sim_scores[query_idx] = -1
    
    ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def recommend_content_with_category(query_idx, top_k=10):
    """
    Model 2:
    Content + Category = title + description + category
    """
    sim_scores = cosine_similarity(tfidf_with_cat[query_idx], tfidf_with_cat).flatten()
    sim_scores[query_idx] = -1
    
    ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def recommend_hybrid(query_idx, top_k=10, w_sim=0.75, w_rating=0.15, w_pop=0.10):
    """
    Model 3:
    Hybrid = content + category + rating + popularity
    """
    sim_scores = cosine_similarity(tfidf_with_cat[query_idx], tfidf_with_cat).flatten()
    sim_scores[query_idx] = -1
    
    final_scores = (
        w_sim * sim_scores +
        w_rating * rating_scores +
        w_pop * popularity_scores
    )
    
    final_scores[query_idx] = -1
    
    ranked_indices = np.argsort(final_scores)[::-1][:top_k]
    return ranked_indices.tolist()

In [11]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    
    if k == 0:
        return 0.0
    
    hit_count = len(set(recommended_k) & relevant)
    return hit_count / k


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return 1.0 if len(set(recommended_k) & relevant) > 0 else 0.0


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    
    dcg = 0.0
    
    for i, item_idx in enumerate(recommended_k):
        if item_idx in relevant:
            dcg += 1.0 / math.log2(i + 2)
    
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


def avg_rating_at_k(recommended, k):
    recommended_k = recommended[:k]
    
    if len(recommended_k) == 0:
        return 0.0
    
    return df.loc[recommended_k, "rating_clean"].mean()


def avg_popularity_at_k(recommended, k):
    recommended_k = recommended[:k]
    
    if len(recommended_k) == 0:
        return 0.0
    
    return np.log1p(df.loc[recommended_k, "rating_count_clean"]).mean()

In [12]:
valid_query_indices = []

for idx in range(len(df)):
    relevant = get_relevant_items(idx)
    
    if len(relevant) >= 3:
        valid_query_indices.append(idx)

print("Số sản phẩm có thể đánh giá:", len(valid_query_indices))

np.random.seed(42)

sample_size = min(300, len(valid_query_indices))
eval_indices = np.random.choice(valid_query_indices, size=sample_size, replace=False)

print("Số sản phẩm dùng để đánh giá:", len(eval_indices))

Số sản phẩm có thể đánh giá: 1225
Số sản phẩm dùng để đánh giá: 300


In [13]:
def evaluate_model(model_name, recommend_func, k=10):
    rows = []
    
    for query_idx in eval_indices:
        relevant = get_relevant_items(query_idx)
        recommended = recommend_func(query_idx, top_k=k)
        
        rows.append({
            "Model": model_name,
            f"Precision@{k}": precision_at_k(recommended, relevant, k),
            f"NDCG@{k}": ndcg_at_k(recommended, relevant, k),
            f"HitRate@{k}": hit_rate_at_k(recommended, relevant, k),
            f"AvgRating@{k}": avg_rating_at_k(recommended, k),
            f"AvgPopularity@{k}": avg_popularity_at_k(recommended, k),
        })
    
    result = pd.DataFrame(rows).groupby("Model").mean().reset_index()
    return result

In [14]:
K = 10

result_content_only = evaluate_model(
    "Content Only",
    recommend_content_only,
    k=K
)

result_content_category = evaluate_model(
    "Content + Category",
    recommend_content_with_category,
    k=K
)

result_hybrid = evaluate_model(
    "Hybrid",
    lambda query_idx, top_k: recommend_hybrid(
        query_idx,
        top_k=top_k,
        w_sim=0.75,
        w_rating=0.15,
        w_pop=0.10
    ),
    k=K
)

metrics_table = pd.concat(
    [
        result_content_only,
        result_content_category,
        result_hybrid
    ],
    ignore_index=True
)

metrics_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Content Only,0.771333,0.836973,0.983333,4.078400,8.353494
1,Content + Category,0.813667,0.875619,0.990000,4.075133,8.352294
2,Hybrid,0.807667,0.872118,0.990000,4.114867,8.724431


In [15]:
metrics_table_rounded = metrics_table.copy()

for col in metrics_table_rounded.columns:
    if col != "Model":
        metrics_table_rounded[col] = metrics_table_rounded[col].round(4)

metrics_table_rounded

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Content Only,0.7713,0.8370,0.9833,4.0784,8.3535
1,Content + Category,0.8137,0.8756,0.9900,4.0751,8.3523
2,Hybrid,0.8077,0.8721,0.9900,4.1149,8.7244


In [16]:
hybrid_configs = [
    ("Hybrid 0.80/0.10/0.10", 0.80, 0.10, 0.10),
    ("Hybrid 0.75/0.15/0.10", 0.75, 0.15, 0.10),
    ("Hybrid 0.70/0.15/0.15", 0.70, 0.15, 0.15),
]

hybrid_results = []

for name, w_sim, w_rating, w_pop in hybrid_configs:
    result = evaluate_model(
        name,
        lambda query_idx, top_k, w_sim=w_sim, w_rating=w_rating, w_pop=w_pop: recommend_hybrid(
            query_idx,
            top_k=top_k,
            w_sim=w_sim,
            w_rating=w_rating,
            w_pop=w_pop
        ),
        k=K
    )
    hybrid_results.append(result)

hybrid_metrics_table = pd.concat(hybrid_results, ignore_index=True)

for col in hybrid_metrics_table.columns:
    if col != "Model":
        hybrid_metrics_table[col] = hybrid_metrics_table[col].round(4)

hybrid_metrics_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Hybrid 0.80/0.10/0.10,0.8083,0.8733,0.99,4.1049,8.6874
1,Hybrid 0.75/0.15/0.10,0.8077,0.8721,0.99,4.1149,8.7244
2,Hybrid 0.70/0.15/0.15,0.8057,0.8709,0.99,4.1214,8.8989


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

tfidf_param_grid = [
    {
        "max_features": 3000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 5000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 5000,
        "ngram_range": (1, 2),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 2),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 2),
        "min_df": 2
    }
]


def evaluate_tfidf_content_only(config, k=10):
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"],
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df["clean_text_no_category"])

    def recommend_func(query_idx, top_k=10):
        sim_scores = cosine_similarity(
            tfidf_matrix[query_idx],
            tfidf_matrix
        ).flatten()

        sim_scores[query_idx] = -1

        ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
        return ranked_indices.tolist()

    result = evaluate_model(
        model_name=(
            f"Content Only | "
            f"max_features={config['max_features']}, "
            f"ngram={config['ngram_range']}, "
            f"min_df={config['min_df']}"
        ),
        recommend_func=recommend_func,
        k=k
    )

    result["max_features"] = config["max_features"]
    result["ngram_range"] = str(config["ngram_range"])
    result["min_df"] = config["min_df"]

    return result


tfidf_content_only_results = []

for config in tfidf_param_grid:
    result = evaluate_tfidf_content_only(config, k=10)
    tfidf_content_only_results.append(result)

tfidf_content_only_table = pd.concat(
    tfidf_content_only_results,
    ignore_index=True
)

for col in tfidf_content_only_table.columns:
    if col not in ["Model", "ngram_range"]:
        tfidf_content_only_table[col] = tfidf_content_only_table[col].round(4)

tfidf_content_only_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content Only | max_features=3000, ngram=(1, 1)...",0.7763,0.8425,0.9867,4.0780,8.3829,3000,"(1, 1)",1
1,"Content Only | max_features=5000, ngram=(1, 1)...",0.7740,0.8394,0.9833,4.0774,8.3981,5000,"(1, 1)",1
2,"Content Only | max_features=5000, ngram=(1, 2)...",0.7713,0.8370,0.9833,4.0784,8.3535,5000,"(1, 2)",1
3,"Content Only | max_features=8000, ngram=(1, 1)...",0.7723,0.8378,0.9833,4.0786,8.4062,8000,"(1, 1)",1
4,"Content Only | max_features=8000, ngram=(1, 2)...",0.7677,0.8330,0.9833,4.0772,8.3275,8000,"(1, 2)",1
5,"Content Only | max_features=8000, ngram=(1, 2)...",0.7700,0.8352,0.9833,4.0796,8.3374,8000,"(1, 2)",2


In [18]:
def evaluate_tfidf_with_category(config, k=10):
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"],
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df["clean_text_with_category"])

    def recommend_func(query_idx, top_k=10):
        sim_scores = cosine_similarity(
            tfidf_matrix[query_idx],
            tfidf_matrix
        ).flatten()

        sim_scores[query_idx] = -1

        ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
        return ranked_indices.tolist()

    result = evaluate_model(
        model_name=(
            f"Content + Category | "
            f"max_features={config['max_features']}, "
            f"ngram={config['ngram_range']}, "
            f"min_df={config['min_df']}"
        ),
        recommend_func=recommend_func,
        k=k
    )

    result["max_features"] = config["max_features"]
    result["ngram_range"] = str(config["ngram_range"])
    result["min_df"] = config["min_df"]

    return result


tfidf_with_category_results = []

for config in tfidf_param_grid:
    result = evaluate_tfidf_with_category(config, k=10)
    tfidf_with_category_results.append(result)

tfidf_with_category_table = pd.concat(
    tfidf_with_category_results,
    ignore_index=True
)

for col in tfidf_with_category_table.columns:
    if col not in ["Model", "ngram_range"]:
        tfidf_with_category_table[col] = tfidf_with_category_table[col].round(4)

tfidf_with_category_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content + Category | max_features=3000, ngram=...",0.8057,0.8712,0.99,4.0762,8.3625,3000,"(1, 1)",1
1,"Content + Category | max_features=5000, ngram=...",0.8073,0.8720,0.99,4.0788,8.3936,5000,"(1, 1)",1
2,"Content + Category | max_features=5000, ngram=...",0.8137,0.8756,0.99,4.0751,8.3523,5000,"(1, 2)",1
3,"Content + Category | max_features=8000, ngram=...",0.8040,0.8684,0.99,4.0798,8.3959,8000,"(1, 1)",1
4,"Content + Category | max_features=8000, ngram=...",0.8103,0.8750,0.99,4.0768,8.3464,8000,"(1, 2)",1
5,"Content + Category | max_features=8000, ngram=...",0.8110,0.8750,0.99,4.0768,8.3362,8000,"(1, 2)",2


In [19]:
best_content_only = tfidf_content_only_table.sort_values(
    by="NDCG@10",
    ascending=False
).head(1)

best_content_category = tfidf_with_category_table.sort_values(
    by="NDCG@10",
    ascending=False
).head(1)

print("Best Content Only:")
display(best_content_only)

print("Best Content + Category:")
display(best_content_category)

Best Content Only:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content Only | max_features=3000, ngram=(1, 1)...",0.7763,0.8425,0.9867,4.078,8.3829,3000,"(1, 1)",1


Best Content + Category:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
2,"Content + Category | max_features=5000, ngram=...",0.8137,0.8756,0.99,4.0751,8.3523,5000,"(1, 2)",1


In [20]:
tfidf_content_only_table.to_csv(
    "tfidf_content_only_tuning.csv",
    index=False
)

tfidf_with_category_table.to_csv(
    "tfidf_with_category_tuning.csv",
    index=False
)

print("Saved:")
print("- tfidf_content_only_tuning.csv")
print("- tfidf_with_category_tuning.csv")

Saved:
- tfidf_content_only_tuning.csv
- tfidf_with_category_tuning.csv
